# Proyecto 2 — Reconocimiento de vocales de LESCO con redes neuronales

Dataset propio · Google Colab · Keras/TensorFlow

Clasifica imágenes de señas **estáticas** del alfabeto dactilológico de **LESCO** (vocales
A, E, I, O, U), comparando **tres enfoques** de complejidad creciente sobre tu propio dataset:

1. **MLP** (red densa) — baseline, aplana la imagen como en el Proyecto 1.
2. **CNN** — red convolucional propia, aprende patrones espaciales.
3. **Transfer learning** (MobileNetV2) — modelo preentrenado, gran rendimiento con pocos datos.

Al final se comparan los tres (accuracy, precision, recall, F1, matriz de confusión), se
contrastan con el SVM del Proyecto 1 y se guarda el mejor modelo (`.keras`).

Las clases se detectan solas a partir de las carpetas del dataset, así que no hace falta
listarlas a mano.

---
### Plan de 3 semanas
- **Semana 1:** recolectar dataset + secciones 1–4 (carga, exploración, augmentation) + MLP.
- **Semana 2:** sección CNN — entrenar, graficar curvas, combatir overfitting, ajustar.
- **Semana 3:** transfer learning + evaluación comparativa + documentación (README / MODEL_CARD).

> **Antes de correr:** Entorno de ejecución → Cambiar tipo de entorno de ejecución → **GPU**.

## 1. Configuración e importaciones

In [ ]:
import os, shutil
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import (confusion_matrix, classification_report,
                             precision_recall_fscore_support, accuracy_score)
import pandas as pd
import seaborn as sns

print("TensorFlow:", tf.__version__)
print("GPU disponible:", "SÍ ✅" if tf.config.list_physical_devices('GPU') else "NO ❌ (activala en Entorno de ejecución)")

# --- Parámetros globales (ajustables) ---
IMG_SIZE   = (128, 128)   # mismo tamaño que el Proyecto 1
BATCH      = 32
SEED       = 42
EPOCHS     = 25
keras.utils.set_random_seed(SEED)


## 2. Conectar el dataset

Tenés **dos maneras** de meter tus fotos. Ambas necesitan la misma estructura: una
**subcarpeta por clase**.

```
dataset/
    clase_a/
    clase_b/
    clase_c/
```

**Opción A — Google Drive (recomendada).** Subí la carpeta `dataset/` a tu Drive, ejecutá
la celda, autorizá el acceso y ajustá `DATA_DIR` si tu ruta es distinta. Queda guardada y
no la tenés que volver a subir en cada sesión.

> **Vos vas a usar la Opción B (ZIP):** saltá esta celda de Drive y usá la de abajo,
> subiendo el `dataset_128.zip` que generaste en el notebook de preparación.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = "/content/drive/MyDrive/proyecto2/dataset"   # <-- ajustá si hace falta

assert os.path.isdir(DATA_DIR), f"No encontré la carpeta {DATA_DIR}"
clases = sorted([d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))])
print("Clases encontradas:", clases)
for c in clases:
    n = len(os.listdir(os.path.join(DATA_DIR, c)))
    print(f"  {c}: {n} imágenes")


### Opción B — Subir el dataset como ZIP

Comprimí tu carpeta `dataset/` en un archivo `.zip` y subilo de una sola vez. La celda lo
descomprime y detecta las clases.

> **Cuidado con dos cosas comunes:**
> - `DATA_DIR` debe apuntar a la carpeta que contiene **directamente** las subcarpetas de clase.
>   Si al comprimir quedó un nivel extra (ej. `dataset/dataset/clase_a/`), ajustá la ruta.
> - Si comprimiste en Mac, puede aparecer una carpeta basura `__MACOSX`; la celda la ignora.

In [ ]:
from google.colab import files
import zipfile, os, shutil

subida = files.upload()                 # elegí tu dataset.zip
nombre_zip = list(subida.keys())[0]

destino = "/content/dataset_subido"
if os.path.isdir(destino):
    shutil.rmtree(destino)              # limpiar si ya existía
with zipfile.ZipFile(nombre_zip, "r") as z:
    z.extractall(destino)

# Ignorar la carpeta basura de Mac si aparece
mac = os.path.join(destino, "__MACOSX")
if os.path.isdir(mac):
    shutil.rmtree(mac)

# Detectar automáticamente la carpeta que contiene las subcarpetas de clase
subdirs = [d for d in os.listdir(destino) if os.path.isdir(os.path.join(destino, d))]
if len(subdirs) == 1:                   # quedó un nivel extra (ej. dataset/)
    DATA_DIR = os.path.join(destino, subdirs[0])
else:
    DATA_DIR = destino

print("DATA_DIR =", DATA_DIR)
clases = sorted([d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))])
print("Clases encontradas:", clases)
for c in clases:
    n = len(os.listdir(os.path.join(DATA_DIR, c)))
    print(f"  {c}: {n} imágenes")


## 3. Cargar y dividir los datos (train / val / test)

`image_dataset_from_directory` lee las imágenes y deduce la etiqueta de la subcarpeta.
Hacemos 70% entrenamiento y 30% restante, que luego partimos en validación y prueba.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

# 1) Cargar TODAS las imágenes sin batch (una por elemento) y en orden fijo
full = keras.utils.image_dataset_from_directory(
    DATA_DIR, labels="inferred", label_mode="int",
    image_size=IMG_SIZE, batch_size=None, shuffle=False)

class_names = full.class_names
num_classes = len(class_names)
TOTAL = full.cardinality().numpy()

# 2) Barajar UNA sola vez, con orden FIJO (mismo en cada época y para cada split).
#    Esto mezcla las clases antes de dividir -> cada split tendrá las 5 vocales.
full = full.shuffle(TOTAL, seed=SEED, reshuffle_each_iteration=False)

# 3) División 70% train / 15% val / 15% test (por número de imágenes)
n_train = int(0.70 * TOTAL)
n_val   = int(0.15 * TOTAL)
train_ds = (full.take(n_train).cache()
            .shuffle(n_train, seed=SEED, reshuffle_each_iteration=True)
            .batch(BATCH).prefetch(AUTOTUNE))
val_ds   = full.skip(n_train).take(n_val).batch(BATCH).cache().prefetch(AUTOTUNE)
test_ds  = full.skip(n_train + n_val).batch(BATCH).cache().prefetch(AUTOTUNE)

print("Clases:", class_names)
print(f"Total: {TOTAL}  |  train: {n_train}  val: {n_val}  test: {TOTAL - n_train - n_val}")

# 4) Verificación clave: cuántas imágenes de cada vocal cayeron en cada split
import collections
def distribucion(ds, nombre):
    cont = collections.Counter()
    for _, y in ds:
        cont.update(y.numpy().tolist())
    print(f"  {nombre:5s}:", {class_names[k]: cont[k] for k in sorted(cont)})

print("Distribución por clase (deben aparecer las 5 vocales en cada uno):")
distribucion(train_ds, "train")
distribucion(val_ds, "val")
distribucion(test_ds, "test")

## 4. Explorar el dataset

In [ ]:
plt.figure(figsize=(10, 6))
for images, labels in train_ds.take(1):
    for i in range(min(9, images.shape[0])):
        plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[labels[i]])
        plt.axis("off")
plt.tight_layout(); plt.show()


## 5. Data augmentation

El Proyecto 1 **no** tenía augmentation. Aquí generamos variaciones (volteos, rotaciones, zoom)
para que el modelo no memorice fondos ni posiciones. Estas capas solo actúan durante el entrenamiento.

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.10),
    layers.RandomZoom(0.10),
    layers.RandomContrast(0.10),
], name="augmentation")

# Vista previa sobre una imagen
for images, _ in train_ds.take(1):
    plt.figure(figsize=(10, 6))
    for i in range(9):
        aug = data_augmentation(tf.expand_dims(images[0], 0))
        plt.subplot(3, 3, i + 1)
        plt.imshow(aug[0].numpy().astype("uint8")); plt.axis("off")
    plt.suptitle("Misma imagen con augmentation"); plt.tight_layout(); plt.show()
    break


## 6. Funciones auxiliares (curvas y evaluación)

In [ ]:
def plot_history(history, titulo):
    h = history.history
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
    ax[0].plot(h["accuracy"], label="train")
    ax[0].plot(h["val_accuracy"], label="val")
    ax[0].set_title(f"{titulo} — Accuracy"); ax[0].set_xlabel("época"); ax[0].legend()
    ax[1].plot(h["loss"], label="train")
    ax[1].plot(h["val_loss"], label="val")
    ax[1].set_title(f"{titulo} — Loss"); ax[1].set_xlabel("época"); ax[1].legend()
    plt.tight_layout(); plt.show()
    gap = h["accuracy"][-1] - h["val_accuracy"][-1]
    estado = "posible overfitting" if gap > 0.10 else "ok"
    print(f"{titulo}: accuracy final  train={h['accuracy'][-1]:.3f}  "
          f"val={h['val_accuracy'][-1]:.3f}  (brecha {gap:+.3f} → {estado})")


resultados = {}

def evaluar(model, ds, nombre):
    # Una sola pasada: recojo etiquetas y predicciones juntas para que queden alineadas
    y_true, y_pred = [], []
    for x, y in ds:
        p = model.predict(x, verbose=0)
        y_true.append(y.numpy())
        y_pred.append(np.argmax(p, axis=1))
    y_true = np.concatenate(y_true)
    y_pred = np.concatenate(y_pred)

    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0)
    resultados[nombre] = {"Accuracy": acc, "Precision": prec,
                          "Recall": rec, "F1": f1,
                          "Parámetros": model.count_params()}
    print(f"{nombre}:  accuracy={acc:.3f}  precision={prec:.3f}  "
          f"recall={rec:.3f}  F1={f1:.3f}")
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(4.6, 3.6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f"Matriz de confusión — {nombre}")
    plt.ylabel("Real"); plt.xlabel("Predicho"); plt.tight_layout(); plt.show()

early = keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)

## 7. Modelo A — MLP (baseline)

Aplana la imagen a un vector y usa capas densas. Es el equivalente neuronal al Proyecto 1:
sirve para mostrar el punto de partida antes de las convoluciones.

In [ ]:
mlp = keras.Sequential([
    layers.Rescaling(1./255, input_shape=IMG_SIZE + (3,)),
    layers.Flatten(),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(128, activation="relu"),
    layers.Dense(num_classes, activation="softmax"),
], name="MLP")

mlp.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
print(f"MLP — parámetros entrenables: {mlp.count_params():,}")
# mlp.summary()   # descomentá para ver la arquitectura completa
hist_mlp = mlp.fit(train_ds, validation_data=val_ds, epochs=EPOCHS,
                   callbacks=[early], verbose=2)
plot_history(hist_mlp, "MLP")
evaluar(mlp, test_ds, "MLP")


## 8. Modelo B — CNN propia

El corazón del proyecto. Bloques de `Conv2D` + `MaxPooling2D` extraen características
espaciales; el `Dropout` y la augmentation combaten el overfitting.

In [ ]:
# CNN final: volvemos a Flatten (la unica que APRENDE de verdad en este dataset)
# y atacamos el overfitting con REGULARIZACION, no bajando la arquitectura:
#  - augmentation suave (rotacion + zoom, sin flip)
#  - L2 en la capa densa + Dropout alto
#  - LR bajo (1e-4) que ya demostro estabilidad
aug_cnn = keras.Sequential([
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.10),
], name="aug_suave")

cnn = keras.Sequential([
    layers.Rescaling(1./255, input_shape=IMG_SIZE + (3,)),
    aug_cnn,

    layers.Conv2D(32, 3, padding="same", use_bias=False),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.MaxPooling2D(),

    layers.Conv2D(64, 3, padding="same", use_bias=False),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.MaxPooling2D(),

    layers.Conv2D(128, 3, padding="same", use_bias=False),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.MaxPooling2D(),

    layers.Flatten(),
    layers.Dense(128, activation="relu",
                 kernel_regularizer=keras.regularizers.l2(1e-4)),
    layers.Dropout(0.5),
    layers.Dense(num_classes, activation="softmax"),
], name="CNN")

cnn.compile(optimizer=keras.optimizers.Adam(1e-4),
            loss="sparse_categorical_crossentropy", metrics=["accuracy"])
print(f"CNN — parámetros entrenables: {cnn.count_params():,}")

early_cnn = keras.callbacks.EarlyStopping(
    monitor="val_accuracy", patience=15, restore_best_weights=True)
reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss", factor=0.5, patience=6, min_lr=1e-6)

hist_cnn = cnn.fit(train_ds, validation_data=val_ds, epochs=60,
                   callbacks=[early_cnn, reduce_lr], verbose=2)
plot_history(hist_cnn, "CNN")
evaluar(cnn, test_ds, "CNN")

## 9. Modelo C — Transfer learning (MobileNetV2)

Reutilizamos un modelo entrenado con millones de imágenes (ImageNet) y solo entrenamos
la capa final para tus clases. MobileNetV2 espera entradas en el rango [-1, 1].

In [ ]:
base = keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,), include_top=False, weights="imagenet")
base.trainable = False   # congelado: solo entrenamos la cabeza

# Augmentation suave propio (sin flip ni contraste, que desestabilizaban)
aug_tl = keras.Sequential([
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.10),
], name="aug_tl")

tl = keras.Sequential([
    layers.Rescaling(1./127.5, offset=-1, input_shape=IMG_SIZE + (3,)),
    aug_tl,
    base,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation="softmax"),
], name="TransferLearning")

tl.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
print(f"Transfer — parámetros entrenables: {sum(w.shape.num_elements() for w in tl.trainable_weights):,}")

# EarlyStopping paciente y sobre val_accuracy: evita cortar a medio entrenar y restaurar pesos malos
early_tl = keras.callbacks.EarlyStopping(
    monitor="val_accuracy", patience=10, restore_best_weights=True)

hist_tl = tl.fit(train_ds, validation_data=val_ds, epochs=30,
                 callbacks=[early_tl], verbose=2)
plot_history(hist_tl, "Transfer Learning")
evaluar(tl, test_ds, "Transfer")

## 10. Comparación final

In [ ]:
# Poné el accuracy que obtuviste con el SVM en el Proyecto 1 (para comparar)
ACC_SVM_P1 = 0.977   # <-- ajustá a tu valor real

# Tabla comparativa (una sola, limpia)
tabla = pd.DataFrame(resultados).T[["Accuracy", "Precision", "Recall", "F1", "Parámetros"]]
vista = tabla.copy()
for col in ["Accuracy", "Precision", "Recall", "F1"]:
    vista[col] = vista[col].map(lambda x: f"{x:.3f}")
vista["Parámetros"] = tabla["Parámetros"].map(lambda x: f"{int(x):,}")
print("Comparación de modelos (set de prueba):\n")
print(vista.to_string())

ganador = tabla["Accuracy"].astype(float).idxmax()
print(f"\nMejor modelo: {ganador}  (accuracy {float(tabla.loc[ganador,'Accuracy']):.3f})")
print(f"Referencia SVM Proyecto 1: accuracy {ACC_SVM_P1:.3f}")

# Gráfico de accuracy, con el SVM del Proyecto 1 como referencia
nombres = list(tabla.index) + ["SVM (P1)"]
accs = [float(a) for a in tabla["Accuracy"]] + [ACC_SVM_P1]
plt.figure(figsize=(6.5, 4))
barras = plt.bar(nombres, accs, color=["#888", "#4c78a8", "#54a24b", "#cc4444"])
plt.ylabel("Accuracy (test)"); plt.ylim(0, 1); plt.title("Comparación de modelos")
for b, v in zip(barras, accs):
    plt.text(b.get_x() + b.get_width()/2, v + 0.01, f"{v:.3f}", ha="center")
plt.tight_layout(); plt.show()

# Detalle por clase SOLO del mejor modelo (para no llenar de tablas)
modelos = {"MLP": mlp, "CNN": cnn, "Transfer": tl}
y_true, y_pred = [], []
for x, y in test_ds:
    y_true.append(y.numpy())
    y_pred.append(np.argmax(modelos[ganador].predict(x, verbose=0), axis=1))
y_true = np.concatenate(y_true); y_pred = np.concatenate(y_pred)
print(f"\nDetalle por clase — {ganador}:")
print(classification_report(y_true, y_pred, target_names=class_names, digits=3))

## 11. Guardar el mejor modelo

In [ ]:
NOMBRE = "C31037_Walther_Barrantes"     # convención con carné, como en el Proyecto 1
mejor = {"MLP": mlp, "CNN": cnn, "Transfer": tl}[ganador]

ruta = f"/content/{NOMBRE}.keras"
mejor.save(ruta)
print(f"Modelo ganador ({ganador}) guardado en:", ruta)

# Si montaste Google Drive (Opción A), guardar también una copia ahí
drive_dir = "/content/drive/MyDrive/proyecto2"
if os.path.isdir("/content/drive/MyDrive"):
    os.makedirs(drive_dir, exist_ok=True)
    shutil.copy(ruta, os.path.join(drive_dir, f"{NOMBRE}.keras"))
    print("Copia en Drive:", os.path.join(drive_dir, f"{NOMBRE}.keras"))

# Descargar el modelo a tu compu (para el repo, carpeta models/)
from google.colab import files
files.download(ruta)

## 12. Usar el modelo con imágenes nuevas (inferencia / demo en vivo)

Cargá el modelo guardado y clasificá cualquier foto de una mano que subas.
Esto sirve tanto para tu set de prueba "real" como para el demo de la presentación.

> **Importante:** las imágenes se pasan con valores 0–255 *sin normalizar*. El modelo ya
> tiene la capa de reescalado incorporada, así que dividir por 255 aquí daría predicciones erróneas.

In [ ]:
from google.colab import files
from tensorflow.keras.utils import load_img, img_to_array

# Cargar el modelo entrenado.
# Si reabrís el notebook en otra sesión, definí estas dos líneas a mano:
#   ruta = "/content/drive/MyDrive/proyecto2/modelo_Transfer.keras"
#   class_names = ['papel', 'piedra', 'tijera']   # orden alfabético, como los carga Keras
modelo_inf = keras.models.load_model(ruta)

print("Subí una o varias fotos de una mano (piedra, papel o tijera):")
subidas = files.upload()

for nombre_archivo in subidas:
    img = load_img(nombre_archivo, target_size=IMG_SIZE)
    x = img_to_array(img)                 # valores 0-255 (el modelo reescala solo)
    x = np.expand_dims(x, axis=0)
    probs = modelo_inf.predict(x, verbose=0)[0]
    pred = class_names[int(np.argmax(probs))]
    conf = 100 * float(np.max(probs))

    plt.figure(figsize=(3, 3))
    plt.imshow(img); plt.axis("off")
    plt.title(f"{pred}  ({conf:.1f}%)")
    plt.show()

    print(f"{nombre_archivo}  →  {pred}  ({conf:.1f}% de confianza)")
    for c, p in zip(class_names, probs):
        print(f"     {c:8s}: {100*p:.1f}%")
    print()


## 13. Para el reporte (checklist)

- [ ] Tabla con cantidad de imágenes por clase y cómo se recolectaron (luz, ángulo, fondo).
- [ ] Curvas de entrenamiento de los 3 modelos y discusión del overfitting.
- [ ] Tabla comparativa MLP vs CNN vs Transfer Learning **vs SVM del Proyecto 1**.
- [ ] Matrices de confusión y análisis de qué clases se confunden.
- [ ] (Bonus) Probar el modelo con el set de prueba "real" tomado en otras condiciones → discutir generalización.
- [ ] README.md y MODEL_CARD.md (misma estructura del Proyecto 1).

**Ideas si te sobra tiempo:** *fine-tuning* (descongelar las últimas capas de MobileNetV2), probar otra arquitectura preentrenada, o visualizar qué "ve" la CNN con mapas de activación (Grad-CAM).
